# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. All references to data entities (record sets, fields, columns) are by their `@id`, per best practice.

### Dataset Source

The dataset is described by a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List all record sets, their IDs, associated fields, and columns by their `@id`.

In [ ]:
# Print all available record sets and their fields
record_sets = dataset.metadata.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"  - RecordSet name: {rs.name}, @id: {rs.id}")
    print(f"    Fields:")
    for fld in rs.fields:
        print(f"      - Field: {fld.name}, @id: {fld.id}, type: {fld.data_type}, column: {fld.column.id if fld.column else 'N/A'}")
    print('')

## 3. Data Extraction
Load data for each record set by `@id` (e.g., `cr:RecordSet/proteins`).

**Note:** Adjust the actual `@id` values from the printed overview if needed.

In [ ]:
# Manually specify the record set @ids based on the above overview if auto-extract not feasible
# Here we use typical biological data record set names, adjust if needed
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame.from_records(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Perform filtering, normalization, and group by operations for a numeric field in a record set.

We'll pick the first record set with numeric fields found in the overview above, e.g., `cr:RecordSet/proteins`, and a numeric field such as `cr:Field/abundance` or `cr:Field/molecular_weight` (update the field `@id` if you see it in the columns from section 3 output).

In [ ]:
# Example: Suppose the main protein table record set is @id 'cr:RecordSet/proteins'
main_record_set_id = None
for rid, df in dataframes.items():
    if 'abundance' in df.columns:
        main_record_set_id = rid
        break
if main_record_set_id is None:
    # fallback: pick first DataFrame
    main_record_set_id = list(dataframes.keys())[0]
    print(f"abundance field not found, using first available record set: {main_record_set_id}")

df = dataframes[main_record_set_id]

# Find numeric field
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    print(f"No numeric fields detected in {main_record_set_id}. EDA limited.")
else:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    # Filtering
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a plausible text/categorical field
    group_field = None
    categoricals = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field]
    if categoricals:
        group_field = categoricals[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field, and relationship with a group field if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric fields available for visualization.")
else:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If we found a group field
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion

- We used `mlcroissant` for programmatic dataset access via Croissant schema `@id`s.
- We examined available record sets, fields, and columns.
- We loaded the main data table and performed filtering, normalization, and basic grouping.
- We visualized the data distributions and relationships between numeric and categorical attributes.

For more advanced or domain-specific analysis, consult the full schema and documentation.